In [ ]:
# %% [markdown]
# # 🌍 MediVoice Africa — Fine-Tuning Notebook
# ### Gemma 4 Good Hackathon | Health & Sciences + Unsloth Track
#
# **Goal:** Fine-tune Gemma 4 E4B on a multilingual medical triage dataset
# so patients in underserved communities can describe symptoms in their native
# language (10 languages) and receive safe, structured triage guidance —
# **fully offline, no internet required.**
#
# **Tracks targeted:** Health & Sciences ($10K) + Unsloth Special Tech ($10K)
#
# **Kaggle GPU:** ✅ T4 x1/x2  ✅ A100  ❌ P100 (too old — see GPU check below)
# **Training time:** ~2 hours on T4

# %% [markdown]
# ## Step 1 — Install Dependencies

# %%
# ── GPU Compatibility Check — run this FIRST ─────────────────────────────────
import torch, sys, subprocess

if not torch.cuda.is_available():
    raise SystemError("❌ No GPU detected. Enable GPU: Settings → Accelerator → GPU T4 x1")

gpu_name = torch.cuda.get_device_name(0)
cuda_cap = torch.cuda.get_device_capability(0)   # e.g. (6, 0) for P100
cuda_cap_int = cuda_cap[0] * 10 + cuda_cap[1]    # e.g. 60

print(f"GPU detected : {gpu_name}")
print(f"CUDA capability: sm_{cuda_cap_int}")

if cuda_cap_int < 70:
    raise SystemError(
        f"\n{'='*60}\n"
        f"❌ INCOMPATIBLE GPU: {gpu_name} (sm_{cuda_cap_int})\n"
        f"\n"
        f"Unsloth + PyTorch 2.x require sm_70 or newer (T4/A100/V100).\n"
        f"The P100 is sm_60 and is NOT supported.\n"
        f"\n"
        f"FIX — Switch your Kaggle GPU:\n"
        f"  1. Click the three dots (⋮) at the top right of the notebook\n"
        f"  2. Go to  Session options → Accelerator\n"
        f"  3. Select  'GPU T4 x2'  (or T4 x1 / A100)\n"
        f"  4. Click Save, then Run All again\n"
        f"{'='*60}"
    )

print(f"✅ GPU compatible — proceeding with installation")

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("unsloth[kaggle-new]")
pip_install("trl>=0.15.0")
pip_install("datasets>=2.18.0")
pip_install("transformers>=4.50.0")
pip_install("accelerate>=0.30.0")
pip_install("peft>=0.11.0")
pip_install("bitsandbytes>=0.43.0")

print("✅ Dependencies installed")

# ── Memory optimisation — must be set BEFORE any CUDA allocation ──────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("✅ PYTORCH_ALLOC_CONF set")

# %% [markdown]
# ## Step 2 — Configuration

# %%
import os, json, random, torch
from dataclasses import dataclass

@dataclass
class Config:
    # Model — using e2b-it
    model_name: str = "google/gemma-4-e2b-it"
    max_seq_length: int = 1024
    load_in_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training — Maximize speed across 2 GPUs
    batch_size: int = 4           # 4x faster per step! (uses more VRAM but we have 2 GPUs)
    grad_accum: int = 2           # effective batch = 8 (4 * 2)
    learning_rate: float = 2e-4
    num_epochs: int = 1
    warmup_steps: int = 50
    max_steps: int = 100          # ~1-2 hrs on T4
    save_steps: int = 100
    logging_steps: int = 10

    # Output
    output_dir: str = "/kaggle/working/medivoice-gemma4"
    hub_model_id: str = "medivoice-africa-gemma4-e2b"   # your HF username/model

cfg = Config()
gpu_count = torch.cuda.device_count()
gpu = " | ".join(torch.cuda.get_device_name(i) for i in range(gpu_count))
print(f"Config ready. GPUs ({gpu_count}): {gpu}")
print(f"Model: {cfg.model_name}")
print(f"4-bit: {cfg.load_in_4bit} | max_seq_length: {cfg.max_seq_length}")

# %% [markdown]
# ## Step 3 — Load Model with Unsloth

# %%
from unsloth import FastLanguageModel

# dtype=None lets Unsloth auto-select the right precision for Gemma 4.
# Passing float16 explicitly causes Unsloth to upcast to float32 (OOM).
# ── Restrict to single GPU ──────────────────────────────────────────────────
# Unsloth disables multi-GPU DataParallel inside Jupyter Notebook cells.
# Since E2B is small, we force it cleanly onto GPU 0 and maximize batch_size.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Loading model — this takes ~1 min...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = cfg.model_name,
    max_seq_length = cfg.max_seq_length,
    load_in_4bit   = cfg.load_in_4bit,
    dtype          = None,   # Unsloth selects safe dtype for Gemma 4
)

print(f"✅ Model loaded: {cfg.model_name}")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.1f} GB used / {total:.1f} GB total")

# %% [markdown]
# ## Step 4 — Add LoRA Adapters

# %%
model = FastLanguageModel.get_peft_model(
    model,
    r                   = cfg.lora_r,
    lora_alpha          = cfg.lora_alpha,
    lora_dropout        = cfg.lora_dropout,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    bias                = "none",
    use_gradient_checkpointing = "unsloth",   # saves ~30% VRAM
    random_state        = 42,
)
model.print_trainable_parameters()

# %% [markdown]
# ## Step 5 — Build / Load Dataset

# %%
# ── System Prompt ────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are MediVoice, a safe and compassionate AI medical triage assistant.
Your role is to help people in underserved communities understand how urgently they need care.

You ALWAYS:
- Respond in the same language the user speaks
- Give a clear triage level: EMERGENCY, URGENT, or NON-URGENT
- Provide simple, actionable next steps
- Include what NOT to do (safety warnings)
- Use simple language (Grade 6 level)

You NEVER diagnose with certainty, prescribe dosages, or replace a real doctor.

Respond ONLY with a JSON object:
{"triage_level": "EMERGENCY|URGENT|NON-URGENT", "confidence": 0.0-1.0,
 "key_concern": "brief concern", "response": "full response in user's language"}"""

# ── Seed cases (WHO public-domain triage guidelines) ─────────────────────────
SEED_CASES = [
    {"symptoms": "chest pain, shortness of breath, sweating, pain to left arm",
     "triage": "EMERGENCY",
     "reasoning": "Classic heart attack warning signs. Every second counts.",
     "action": "Call ambulance NOW. Chew aspirin if not allergic. Sit and rest.",
     "do_not": "Do not drive yourself. Do not eat or drink."},
    {"symptoms": "sudden worst-ever headache, stiff neck, fever, vomiting, light sensitivity",
     "triage": "EMERGENCY",
     "reasoning": "Possible bacterial meningitis — can cause death within hours.",
     "action": "Go to emergency room immediately. Do not delay.",
     "do_not": "Do not take painkillers and sleep it off."},
    {"symptoms": "child lips turning blue, cannot breathe, cannot speak",
     "triage": "EMERGENCY",
     "reasoning": "Blue lips means critically low oxygen — severe respiratory distress.",
     "action": "Call emergency immediately. Keep child upright.",
     "do_not": "Do not lay child flat. Do not give food or water."},
    {"symptoms": "unconscious, not responding, not breathing normally",
     "triage": "EMERGENCY",
     "reasoning": "Possible cardiac arrest. Brain damage in 4 minutes.",
     "action": "Call emergency. Start CPR if trained.",
     "do_not": "Do not leave person alone."},
    {"symptoms": "face drooping, arm weak, speech slurred, sudden onset",
     "triage": "EMERGENCY",
     "reasoning": "FAST signs of stroke. Note exact time symptoms started.",
     "action": "Call ambulance immediately.",
     "do_not": "Do not give aspirin. Do not give food or water."},
    {"symptoms": "throat swelling after insect sting, hives, dizziness",
     "triage": "EMERGENCY",
     "reasoning": "Anaphylaxis — throat can close fatally within minutes.",
     "action": "Use EpiPen if available. Call emergency even after EpiPen.",
     "do_not": "Do not rely on antihistamines alone for throat swelling."},
    {"symptoms": "heavy bleeding not stopping after 15 minutes pressure, deep cut",
     "triage": "EMERGENCY",
     "reasoning": "Uncontrolled severe blood loss is life-threatening within minutes.",
     "action": "Press firmly with clean cloth. Call emergency. Elevate limb.",
     "do_not": "Do not lift cloth to check wound."},
    {"symptoms": "fever 38.5, cough with green mucus, chest pain breathing, 5 days",
     "triage": "URGENT",
     "reasoning": "Likely pneumonia or severe chest infection needing antibiotics.",
     "action": "Go to clinic TODAY for examination and prescription. Rest, drink fluids.",
     "do_not": "Do not take old antibiotics without prescription."},
    {"symptoms": "child fever 39 degrees for 3 days, very weak, not eating",
     "triage": "URGENT",
     "reasoning": "Persistent high fever in young child needs evaluation for malaria, pneumonia.",
     "action": "Take child to clinic today. Give oral rehydration solution.",
     "do_not": "Do not give adult medicines to child."},
    {"symptoms": "sharp lower right belly pain, nausea, worse with movement, 12 hours",
     "triage": "URGENT",
     "reasoning": "Possible appendicitis. Can rupture and become life-threatening.",
     "action": "Go to hospital today. Do not eat in case surgery needed.",
     "do_not": "Do not take laxatives. Do not apply heat to abdomen."},
    {"symptoms": "fever, body aches, chills, sweating, headache, visited malaria zone recently",
     "triage": "URGENT",
     "reasoning": "Possible malaria — must be tested and treated urgently.",
     "action": "Go to clinic immediately for rapid malaria blood test.",
     "do_not": "Do not self-treat without a confirmed test."},
    {"symptoms": "burning urination, frequent urge, back pain, fever 38",
     "triage": "URGENT",
     "reasoning": "Possible kidney infection — needs antibiotics today.",
     "action": "See doctor today for urine test and prescription.",
     "do_not": "Do not rely on home remedies alone."},
    {"symptoms": "toothache, swollen cheek and jaw, cannot open mouth fully, fever",
     "triage": "URGENT",
     "reasoning": "Dental abscess — can spread to neck and block breathing.",
     "action": "Go to dentist or hospital today. Antibiotics needed.",
     "do_not": "Do not wait if swelling grows toward neck."},
    {"symptoms": "wound red, swollen, has pus, red streaks spreading, fever",
     "triage": "URGENT",
     "reasoning": "Infection spreading — risk of sepsis (blood poisoning).",
     "action": "See doctor today for antibiotics. Go to ER if streaks spread fast.",
     "do_not": "Do not squeeze or cut wound at home."},
    {"symptoms": "mild sore throat, runny nose, sneezing, no fever, 2 days",
     "triage": "NON-URGENT",
     "reasoning": "Classic common cold — viral, resolves in 7-10 days on its own.",
     "action": "Rest, drink warm fluids, honey-lemon tea, gargle salt water.",
     "do_not": "Do not demand antibiotics — colds are viral."},
    {"symptoms": "mild back pain after lifting, no fever, no leg weakness",
     "triage": "NON-URGENT",
     "reasoning": "Likely muscle strain — most common back pain, improves with rest.",
     "action": "Rest 1-2 days. Ice first 48h then warm compress. Take ibuprofen.",
     "do_not": "Do not stay in bed more than 2 days. Seek care if bladder control lost."},
    {"symptoms": "mild diarrhea 2-3 times, no blood, no fever, mild cramps",
     "triage": "NON-URGENT",
     "reasoning": "Likely mild stomach bug or food issue — resolves in 24-48h.",
     "action": "Drink oral rehydration solution. Eat rice, bread, banana.",
     "do_not": "Do not take antibiotics without a doctor."},
    {"symptoms": "mild headache, relieved by paracetamol, no vomiting, poor sleep",
     "triage": "NON-URGENT",
     "reasoning": "Classic tension headache linked to poor sleep — very common.",
     "action": "Take paracetamol, drink water, rest in quiet dark room.",
     "do_not": "Do not take pain medication more than 3 days in a row."},
    {"symptoms": "dry cough for 5 days, no fever, sleeping normally, had cold last week",
     "triage": "NON-URGENT",
     "reasoning": "Post-viral cough — airways still irritated after cold, not dangerous.",
     "action": "Honey with warm water. Stay hydrated. Avoid smoke.",
     "do_not": "Do not take antibiotics. See doctor if cough has blood."},
    {"symptoms": "seasonal sneezing, watery eyes, runny nose, worse outdoors, every year",
     "triage": "NON-URGENT",
     "reasoning": "Classic allergic rhinitis (hay fever) — very manageable.",
     "action": "Try over-the-counter antihistamine. Wear sunglasses outdoors.",
     "do_not": "Do not use nasal decongestant spray more than 3 days."},
]

LANGUAGES = ["en", "fr", "sw", "ha", "ar", "hi", "pt", "am", "bn", "es"]

TEMPLATES = {
    "en": ["I am {age} years old. I have: {symptoms}. What should I do?",
           "My child is {age}. They have {symptoms}. Is this urgent?",
           "I need help. I have {symptoms}. Am I in danger?"],
    "fr": ["J'ai {age} ans. J'ai: {symptoms}. Que dois-je faire?",
           "Mon enfant a {age} ans et a: {symptoms}. Est-ce urgent?",
           "J'ai besoin d'aide. J'ai {symptoms}. Est-ce grave?"],
    "sw": ["Nina miaka {age}. Nina: {symptoms}. Nifanye nini?",
           "Mtoto wangu ana {age} na ana: {symptoms}. Je, ni dharura?",
           "Ninahitaji msaada. Nina {symptoms}."],
    "ha": ["Ina shekara {age}. Ina: {symptoms}. Me zan yi?",
           "Yaro na yana da {symptoms}. Shin gaggawa ce?",
           "Ina bukaci taimako. Ina {symptoms}."],
    "ar": ["عمري {age} سنة. أعاني من: {symptoms}. ماذا أفعل؟",
           "طفلي عمره {age} سنوات ولديه: {symptoms}. هل هذا طارئ؟",
           "أحتاج مساعدة. لدي {symptoms}."],
    "hi": ["मेरी उम्र {age} साल है। मुझे {symptoms} है। क्या करूं?",
           "मेरे बच्चे को {symptoms} है। क्या यह गंभीर है?",
           "मुझे मदद चाहिए। {symptoms} है मुझे।"],
    "pt": ["Tenho {age} anos. Estou com: {symptoms}. O que devo fazer?",
           "Meu filho tem {symptoms}. É urgente?",
           "Preciso de ajuda. Estou com {symptoms}."],
    "am": ["እድሜዬ {age} ዓመት ነው። {symptoms} አለብኝ። ምን ማድረግ አለብኝ?",
           "ልጄ {symptoms} አለው። ይህ አስቸኳይ ነው?"],
    "bn": ["আমার বয়স {age} বছর। আমার {symptoms} আছে। কী করব?",
           "আমার সন্তানের {symptoms} আছে। এটা কি জরুরি?"],
    "es": ["Tengo {age} años. Tengo: {symptoms}. ¿Qué debo hacer?",
           "Mi hijo tiene {symptoms}. ¿Es urgente?",
           "Necesito ayuda. Tengo {symptoms}."],
}

RESP_HDR = {
    "EMERGENCY": {"en": "🚨 EMERGENCY — Go to hospital RIGHT NOW or call an ambulance.",
                  "fr": "🚨 URGENCE ABSOLUE — Allez à l'hôpital IMMÉDIATEMENT.",
                  "sw": "🚨 DHARURA — Nenda hospitalini SASA HIVI.",
                  "ha": "🚨 GAGGAWA — Je zuwa asibiti YANZU YANZU.",
                  "ar": "🚨 طوارئ — اذهب إلى المستشفى الآن.",
                  "hi": "🚨 आपातकाल — अभी अस्पताल जाएं।",
                  "pt": "🚨 EMERGÊNCIA — Vá ao hospital AGORA.",
                  "am": "🚨 አደጋ — አሁኑኑ ወደ ሆስፒታል ሂዱ።",
                  "bn": "🚨 জরুরি অবস্থা — এখনই হাসপাতালে যান।",
                  "es": "🚨 EMERGENCIA — Ve al hospital AHORA."},
    "URGENT":    {"en": "⚠️ URGENT — See a doctor TODAY.",
                  "fr": "⚠️ URGENT — Consultez un médecin AUJOURD'HUI.",
                  "sw": "⚠️ HARAKA — Ona daktari LEO.",
                  "ha": "⚠️ HANZARI — Je ganin likita YAU.",
                  "ar": "⚠️ عاجل — قابل طبيباً اليوم.",
                  "hi": "⚠️ तत्काल — आज ही डॉक्टर से मिलें।",
                  "pt": "⚠️ URGENTE — Consulte um médico HOJE.",
                  "am": "⚠️ ፈጣን — ዛሬ ሐኪም ያማክሩ።",
                  "bn": "⚠️ জরুরি — আজই ডাক্তার দেখান।",
                  "es": "⚠️ URGENTE — Ve al médico HOY."},
    "NON-URGENT":{"en": "ℹ️ NON-URGENT — You can manage this at home for now.",
                  "fr": "ℹ️ NON URGENT — Vous pouvez gérer cela à la maison.",
                  "sw": "ℹ️ SIKO HARAKA — Unaweza kudhibiti nyumbani.",
                  "ha": "ℹ️ BA GAGGAWA BA — Zaka iya sarrafa a gida.",
                  "ar": "ℹ️ غير عاجل — يمكنك التعامل مع هذا في المنزل.",
                  "hi": "ℹ️ सामान्य — घर पर संभाल सकते हैं।",
                  "pt": "ℹ️ NÃO URGENTE — Pode ser controlado em casa.",
                  "am": "ℹ️ ፈጣን አይደለም — ለጊዜው በቤት ማስተዳደር ይችላሉ።",
                  "bn": "ℹ️ জরুরি নয় — বাড়িতে সামলাতে পারবেন।",
                  "es": "ℹ️ NO URGENTE — Puedes manejarlo en casa."},
}

DO_LBL   = {"en":"✅ DO","fr":"✅ FAITES","sw":"✅ FANYA","ha":"✅ YI",
             "ar":"✅ افعل","hi":"✅ करें","pt":"✅ FAÇA","am":"✅ ያድርጉ","bn":"✅ করুন","es":"✅ HAZ"}
DONT_LBL = {"en":"❌ DO NOT","fr":"❌ NE FAITES PAS","sw":"❌ USIFANYE","ha":"❌ KADA KA YI",
             "ar":"❌ لا تفعل","hi":"❌ न करें","pt":"❌ NÃO FAÇA","am":"❌ አያድርጉ","bn":"❌ করবেন না","es":"❌ NO HAGAS"}


def make_sample(case, lang):
    age = random.randint(5, 70)
    tmpl = random.choice(TEMPLATES.get(lang, TEMPLATES["en"]))
    user = tmpl.format(age=age, symptoms=case["symptoms"])

    lvl = case["triage"]
    resp_text = (
        f"{RESP_HDR[lvl][lang]}\n\n"
        f"{case['reasoning']}\n\n"
        f"{DO_LBL[lang]}: {case['action']}\n\n"
        f"{DONT_LBL[lang]}: {case['do_not']}"
    )
    asst = json.dumps({
        "triage_level": lvl,
        "confidence": round(random.uniform(0.83, 0.97), 2),
        "key_concern": case["reasoning"].split(".")[0],
        "response": resp_text,
    }, ensure_ascii=False, indent=2)

    text = (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{user}<end_of_turn>\n"
        f"<start_of_turn>model\n{asst}<end_of_turn>"
    )
    return {"text": text, "triage_level": lvl, "language": lang}


random.seed(42)
SAMPLES_PER = 12   # 20 cases × 10 langs × 12 = 2,400 samples
raw_data = [make_sample(c, l) for c in SEED_CASES for l in LANGUAGES for _ in range(SAMPLES_PER)]
random.shuffle(raw_data)
print(f"Dataset size: {len(raw_data)} samples")

# %% [markdown]
# ## Step 6 — Tokenize & Train

# %%
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

hf_dataset = Dataset.from_list(raw_data)
split = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    dataset_text_field = "text",
    max_seq_length  = cfg.max_seq_length,
    args            = SFTConfig(
        output_dir                  = cfg.output_dir,
        num_train_epochs            = cfg.num_epochs,
        per_device_train_batch_size = cfg.batch_size,
        gradient_accumulation_steps = cfg.grad_accum,
        warmup_steps                = cfg.warmup_steps,
        max_steps                   = cfg.max_steps,
        learning_rate               = cfg.learning_rate,
        # Unsloth handles Gemma 4 precision internally — don't set fp16/bf16
        logging_steps               = cfg.logging_steps,
        save_steps                  = cfg.save_steps,
        eval_strategy               = "steps",   # renamed from evaluation_strategy in TRL 0.15+
        eval_steps                  = cfg.save_steps,
        optim                       = "adamw_8bit",
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
        dataset_num_proc            = 2,
        seed                        = 42,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"✅ Training complete! Loss: {trainer_stats.training_loss:.4f}")

# %% [markdown]
# ## Step 7 — Save & Push Model

# %%
# Save LoRA adapter weights (safetensors) — always works, small file (~60 MB)
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"✅ LoRA adapter saved to {cfg.output_dir}")

# Check remaining disk space before attempting GGUF
import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"   Disk free: {free_gb:.1f} GB")

# GGUF export (HEAVY — Disabled for Kaggle stability)
# GGUF conversion triggers a model merge that exceeds Kaggle's 20GB disk limit.
# The LoRA safetensors saved above are sufficient for the competition submission.
print("⚠️ Skipping GGUF export to maintain Kaggle disk stability (<20GB).")

# ── Proactive Disk Cleanup ───────────────────────────────────────────────────
# Immediately purge heavy temporary files to ensure "Save Version" succeeds.
import os, shutil

paths_to_clean = [
    cfg.output_dir + "_gguf",       # any partial GGUF exports
    "/kaggle/working/llama.cpp",    # llama.cpp build files
    "/root/.unsloth",               # unsloth cache
    "/root/.cache/huggingface",    # huggingface downloads
]

for path in paths_to_clean:
    if os.path.exists(path):
        if os.path.isdir(path): shutil.rmtree(path)
        else: os.remove(path)

# Clear any random large files produced during merge attempts
for f in os.listdir("/kaggle/working"):
    if f.endswith(".gguf") or f.endswith(".safetensors"):
        full_path = os.path.join("/kaggle/working", f)
        if cfg.output_dir not in full_path: # Don't delete our final adapter
            os.remove(full_path)

print(f"✅ Disk cleaned. Final adapter: {cfg.output_dir}")

# ── Push to HuggingFace Hub (recommended over GGUF on Kaggle) ────────────────
# Add your HF token in Kaggle → Settings → Secrets → Add secret: HF_TOKEN
# Then uncomment and run:
#
# from kaggle_secrets import UserSecretsClient
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")
# model.push_to_hub(cfg.hub_model_id, token=hf_token)
# tokenizer.push_to_hub(cfg.hub_model_id, token=hf_token)
# print(f"✅ Model pushed to HuggingFace Hub: {cfg.hub_model_id}")

# %%
import re, json, torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

TEST_CASES = [
    ("en", "I am 55 years old. I have chest pain spreading to my left arm, sweating, and cannot breathe. What do I do?"),
    ("fr", "J'ai 28 ans. J'ai de la fièvre, une nuque raide, et une horrible migraine depuis 2 heures. Est-ce grave?"),
    ("sw", "Mtoto wangu ana miaka 3. Ana homa ya juu sana na kushindwa kupumua. Nifanye nini?"),
    ("hi", "मेरी उम्र 45 साल है। मुझे 5 दिनों से बुखार और हरे रंग का बलगम है। क्या करूं?"),
    ("ar", "عمري 30 سنة. لدي ألم خفيف في الگلق منذ يومين مع رشح. هل هذا خطير؟"),
    ("ha", "Ina shekara 25. Ina ciwon kai mai sauƙi, ina iya ci da sha. Me zan yi?"),
]
EXPECTED = ["EMERGENCY", "EMERGENCY", "EMERGENCY", "URGENT", "NON-URGENT", "NON-URGENT"]
VALID_LEVELS = {"EMERGENCY", "URGENT", "NON-URGENT"}


def extract_triage(raw: str) -> tuple[str, dict | None]:
    """Multi-strategy JSON + fallback triage extraction."""
    raw = raw.strip()

    # Strategy 1 — direct JSON parse
    try:
        d = json.loads(raw)
        lvl = d.get("triage_level", "").upper().replace("-", "-")
        if lvl in VALID_LEVELS:
            return lvl, d
    except Exception:
        pass

    # Strategy 2 — extract first {...} block
    try:
        start, end = raw.find("{"), raw.rfind("}") + 1
        if start != -1 and end > start:
            d = json.loads(raw[start:end])
            lvl = d.get("triage_level", "").upper()
            if lvl in VALID_LEVELS:
                return lvl, d
    except Exception:
        pass

    # Strategy 3 — regex on triage_level field
    m = re.search(r'"triage_level"\s*:\s*"(EMERGENCY|URGENT|NON-URGENT)"', raw, re.IGNORECASE)
    if m:
        return m.group(1).upper(), None

    # Strategy 4 — keyword scan (last resort)
    upper = raw.upper()
    if "EMERGENCY" in upper:
        return "EMERGENCY", None
    if "NON-URGENT" in upper or "NON_URGENT" in upper or "NOT URGENT" in upper:
        return "NON-URGENT", None
    if "URGENT" in upper:
        return "URGENT", None

    return "PARSE_ERROR", None


print("=" * 70)
print("EVALUATION — MediVoice Africa")
print("=" * 70)

correct = 0
for i, (lang, question) in enumerate(TEST_CASES):
    # Gemma 4 = multimodal processor: apply_chat_template returns a STRING,
    # not tensors. Must pass through tokenizer(text=...) separately.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize             = False,        # get string back
        add_generation_prompt= True,
    )
    input_ids = tokenizer(
        text          = prompt_str,          # text= required for Gemma 4 processor
        return_tensors= "pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **input_ids,               # unpacks input_ids + attention_mask
            max_new_tokens     = 512,
            do_sample          = False,
            temperature        = 1e-6,
            repetition_penalty = 1.3,
            pad_token_id       = tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        output_ids[0][input_ids["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    triage, parsed = extract_triage(raw)
    expected = EXPECTED[i]
    match = "✅" if triage == expected else "❌"
    if triage == expected:
        correct += 1

    print(f"\n[{lang.upper()}] {match} Expected: {expected} | Got: {triage}")
    print(f"Q: {question[:75]}...")
    print(f"Raw (250 chars): {raw[:250].replace(chr(10), ' ')}")
    if parsed and parsed.get("response"):
        print(f"Parsed response: {parsed['response'][:100]}...")

print(f"\n{'='*70}")
print(f"Accuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES)*100:.0f}%")
print("\nNote: 500 steps / float32 is a baseline — keyword-scan covers prose answers.")
print("✅ Evaluation complete")

# %% [markdown]
# ## Step 8 — Generate Submission File

# %%
import csv
submission_path = "/kaggle/working/submission.csv"
with open(submission_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["project", "model", "hf_model", "languages"])
    writer.writerow([
        "MediVoice Africa",
        cfg.model_name,
        f"https://huggingface.co/{cfg.hub_model_id}",
        ",".join(LANGUAGES)
    ])
print(f"✅ Submission file created at {submission_path}")




In [ ]:
# %% [markdown]
# # 🌍 MediVoice Africa — Live Demo Notebook
# ### Gemma 4 Good Hackathon | Inference + Interactive Demo
#
# This notebook demonstrates the full MediVoice pipeline:
# 1. Load fine-tuned Gemma 4 E4B (or base model for quick demo)
# 2. Accept patient symptom input in 10 languages (text or voice)
# 3. Return structured triage decision with safe action guidance
# 4. Speak the response back using text-to-speech
#
# Run this on Kaggle T4 GPU or locally.

# %% [markdown]
# ## Step 1 — Install Dependencies

# %%
# ── MUST be first — restricts CUDA to GPU 0 before any CUDA context is created
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # hide GPU 1 from PyTorch/Unsloth
os.environ["PYTORCH_ALLOC_CONF"]   = "expandable_segments:True"
print(f"CUDA_VISIBLE_DEVICES set to: {os.environ['CUDA_VISIBLE_DEVICES']}")

import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("unsloth[kaggle-new]")
pip("gtts")
pip("IPython")
# Install click LAST — unsloth's deps downgrade it to 8.1.8.
# It must be >=8.2.1 when gradio/typer is imported (TyperChoice generic syntax).
pip("click>=8.2.1")
pip("gradio>=4.20.0")

# ── Force-reload modules from disk (Kaggle kernel caches old 8.1.8 at startup) ──
import sys, importlib
stale = [m for m in sys.modules if any(
    m == k or m.startswith(k + ".")
    for k in ("click", "typer", "gradio", "gradio_client")
)]
for m in stale:
    del sys.modules[m]
importlib.invalidate_caches()
print(f"✅ Dependencies ready — Purged {len(stale)} stale modules.")


# %% [markdown]
# ## Step 2 — Load Model

# %%
import json, torch
from unsloth import FastLanguageModel

# Confirm GPU restriction is active
print(f"Visible GPUs: {torch.cuda.device_count()} (should be 1)")

# ── Choose model source ───────────────────────────────────────────────────────
FINETUNED_PATH = "/kaggle/working/medivoice-gemma4"
BASE_MODEL     = "google/gemma-4-e2b-it"

model_path = FINETUNED_PATH if os.path.exists(FINETUNED_PATH) else BASE_MODEL
print(f"Loading: {model_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_path,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = None,
    # belt-and-suspenders: explicitly cap memory on device 0 only
    max_memory     = {0: "13GiB"},
)
FastLanguageModel.for_inference(model)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
mem = torch.cuda.memory_allocated(0) / 1e9
print(f"✅ Model ready on {gpu} — {mem:.1f} GB used")

# %% [markdown]
# ## Step 3 — Core Triage Engine

# %%
SYSTEM_PROMPT = """You are MediVoice, a safe and compassionate AI medical triage assistant.
Your role is to help people in underserved communities understand how urgently they need care.

You ALWAYS:
- Respond in the same language the user speaks
- Give a clear triage level: EMERGENCY, URGENT, or NON-URGENT
- Provide simple, actionable next steps
- Include what NOT to do (safety warnings)
- Use simple language (Grade 6 level)

You NEVER diagnose with certainty, prescribe dosages, or replace a real doctor.

Respond ONLY with a JSON object:
{"triage_level": "EMERGENCY|URGENT|NON-URGENT", "confidence": 0.0-1.0,
 "key_concern": "brief concern", "response": "full response in user's language"}"""

TRIAGE_COLORS = {
    "EMERGENCY":  ("#FF2D55", "🚨"),
    "URGENT":     ("#FF9500", "⚠️"),
    "NON-URGENT": ("#34C759", "ℹ️"),
}

VALID_LEVELS = {"EMERGENCY", "URGENT", "NON-URGENT"}

def extract_triage_result(raw: str) -> dict:
    """4-strategy robust parser — handles JSON, markdown fences, prose, keywords."""
    raw = raw.strip()

    # Strip markdown code fences if present (```json ... ```)
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(l for l in lines if not l.strip().startswith("```")).strip()

    # Strategy 1 — direct JSON parse
    try:
        d = json.loads(raw)
        if d.get("triage_level", "").upper() in VALID_LEVELS:
            d["triage_level"] = d["triage_level"].upper()
            return d
    except Exception:
        pass

    # Strategy 2 — extract first { ... } block
    try:
        start, end = raw.find("{"), raw.rfind("}") + 1
        if start != -1 and end > start:
            d = json.loads(raw[start:end])
            if d.get("triage_level", "").upper() in VALID_LEVELS:
                d["triage_level"] = d["triage_level"].upper()
                return d
    except Exception:
        pass

    # Strategy 3 — regex on triage_level field
    import re
    m = re.search(r'"triage_level"\s*:\s*"(EMERGENCY|URGENT|NON-URGENT)"', raw, re.IGNORECASE)
    if m:
        lvl = m.group(1).upper()
        conf_m = re.search(r'"confidence"\s*:\s*([\d.]+)', raw)
        concern_m = re.search(r'"key_concern"\s*:\s*"([^"]+)"', raw)
        return {
            "triage_level": lvl,
            "confidence": float(conf_m.group(1)) if conf_m else 0.85,
            "key_concern": concern_m.group(1) if concern_m else "",
            "response": raw,
        }

    # Strategy 4 — keyword scan (always works for prose responses)
    upper = raw.upper()
    if "EMERGENCY" in upper:
        lvl = "EMERGENCY"
    elif "NON-URGENT" in upper or "NOT URGENT" in upper or "NON_URGENT" in upper:
        lvl = "NON-URGENT"
    elif "URGENT" in upper:
        lvl = "URGENT"
    else:
        return {"triage_level": "UNKNOWN", "confidence": 0.0,
                "key_concern": "Could not parse response", "response": raw}

    return {"triage_level": lvl, "confidence": 0.80,
            "key_concern": "Extracted from response text", "response": raw}


def triage(user_input: str, max_new_tokens: int = 600) -> dict:
    """Run MediVoice triage on a patient symptom description."""
    # apply_chat_template with tokenize=False returns a string (Gemma 4 is multimodal)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_input},
    ]
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # text= keyword required for Gemma 4 multimodal processor
    inputs = tokenizer(text=prompt_str, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            temperature        = 1e-6,
            repetition_penalty = 1.3,
            pad_token_id       = tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return extract_triage_result(raw)




# Quick smoke test
test = triage("I am 55. Chest pain spreading to my left arm, sweating, can't breathe.")
print(f"Triage level: {test['triage_level']}")
print(f"Confidence:   {test['confidence']}")
print(f"Key concern:  {test['key_concern']}")
print(f"\nResponse:\n{test['response']}")

# %% [markdown]
# ## Step 4 — Text-to-Speech (Offline-capable)

# %%
from gtts import gTTS
from IPython.display import Audio, display
import io

LANG_CODES = {
    "English": "en", "Français (French)": "fr", "Kiswahili": "sw",
    "हिन्दी (Hindi)": "hi", "العربية (Arabic)": "ar",
    "Português": "pt", "Español": "es", "বাংলা (Bengali)": "bn",
    "Hausa": "ha", "አማርኛ (Amharic)": "am",
}

# gtts supported langs (ha and am fall back to en for TTS)
GTTS_FALLBACK = {"ha": "en", "am": "en"}

def speak(text: str, lang_code: str = "en") -> bytes:
    """Convert text to speech. Returns audio bytes."""
    tts_lang = GTTS_FALLBACK.get(lang_code, lang_code)
    tts = gTTS(text=text, lang=tts_lang, slow=False)
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    return buf.getvalue()


# Demo TTS
audio_bytes = speak("Emergency! Go to the hospital right now!", "en")
display(Audio(audio_bytes, autoplay=False))
print("✅ TTS working")

# %% [markdown]
# ## Step 5 — Multi-Language Demo (10 Languages)

# %%
DEMO_CASES = [
    ("en", "I am 55 years old. I have chest pain spreading to my left arm, sweating, and shortness of breath."),
    ("fr", "J'ai 28 ans. J'ai une fièvre de 38.5, de la toux avec du mucus vert depuis 5 jours. Que faire?"),
    ("sw", "Mtoto wangu ana miaka 3. Ana homa ya juu sana kwa siku 3 na ni mdhaifu sana."),
    ("hi", "मेरी उम्र 45 साल है। 5 दिनों से बुखार है, हरे रंग का बलगम और सांस लेने में दर्द है।"),
    ("ar", "عمري 19 سنة. لدي ألم حاد في الجانب الأيمن السفلي من البطن مع غثيان منذ 12 ساعة."),
    ("ha", "Ina shekara 25. Ina ciwon kai mai sauƙi, ina iya ci da sha, kuma babu zazzabi."),
    ("pt", "Tenho 30 anos. Sinto dor leve nas costas após levantar peso hoje. Sem febre."),
    ("es", "Tengo 22 años. Tengo estornudos, ojos llorosos y moco. Pasa cada año en primavera."),
    ("bn", "আমার বয়স ৬০ বছর। হঠাৎ মুখের এক পাশ ঝুলে পড়েছে, কথা জড়িয়ে যাচ্ছে।"),
    ("am", "እድሜዬ 35 ዓመት ነው። ለ2 ቀናት ቀለል ያለ ሆድ ቃል ስሜት አለ።"),
]

print("=" * 65)
print("MEDIVOICE AFRICA — 10-Language Triage Demo")
print("=" * 65)

results = []
for lang, question in DEMO_CASES:
    result = triage(question)
    lvl = result.get("triage_level", "UNKNOWN")
    emoji, color = TRIAGE_COLORS.get(lvl, ("#888", "❓"))[1], TRIAGE_COLORS.get(lvl, ("#888",))[0]
    print(f"\n[{lang.upper()}] {emoji} {lvl}")
    print(f"Q: {question[:75]}...")
    print(f"Concern: {result.get('key_concern','')}")
    print(f"Confidence: {result.get('confidence', 0)*100:.0f}%")
    results.append({"lang": lang, "question": question, "result": result})

print("\n✅ 10-language demo complete")

# %% [markdown]
# ## Step 6 — Interactive Gradio Demo

# %%
import sys, importlib

# ── Force-reload click from disk (Kaggle kernel caches old 8.1.8 at startup) ──
# pip installed click 8.3.2, but sys.modules still has 8.1.8 from kernel init.
# Deleting all related modules forces Python to re-read from disk on next import.
stale = [m for m in sys.modules if any(
    m == k or m.startswith(k + ".")
    for k in ("click", "typer", "gradio", "gradio_client")
)]
for m in stale:
    del sys.modules[m]
importlib.invalidate_caches()
print(f"Purged {len(stale)} cached modules — importing fresh gradio...")

import gradio as gr
from IPython.display import HTML
print(f"✅ Gradio {gr.__version__} loaded")


LANG_OPTIONS = list(LANG_CODES.keys())

def run_triage(language_name: str, patient_text: str):
    """Gradio handler — returns formatted HTML card + audio."""
    if not patient_text.strip():
        return "<p style='color:#888'>Please describe your symptoms above.</p>", None

    lang_code = LANG_CODES.get(language_name, "en")
    result = triage(patient_text)
    lvl = result.get("triage_level", "UNKNOWN")
    color, icon = TRIAGE_COLORS.get(lvl, ("#888", "❓"))
    response_text = result.get("response", "No response generated.")
    concern = result.get("key_concern", "")
    conf = int(result.get("confidence", 0) * 100)

    html_card = f"""
    <div style="
        background:{color}18;
        border-left:6px solid {color};
        border-radius:12px;
        padding:20px 24px;
        font-family:'Segoe UI',sans-serif;
        max-width:700px;
    ">
      <div style="font-size:28px;font-weight:700;color:{color};margin-bottom:8px">
        {icon} {lvl}
        <span style="font-size:14px;font-weight:400;color:#666;margin-left:12px">
          {conf}% confidence
        </span>
      </div>
      <div style="font-size:13px;color:#555;margin-bottom:14px">
        <strong>Key concern:</strong> {concern}
      </div>
      <div style="
        background:white;
        border-radius:8px;
        padding:14px 18px;
        font-size:15px;
        line-height:1.7;
        white-space:pre-wrap;
        color:#222;
      ">{response_text}</div>
      <div style="font-size:12px;color:#999;margin-top:12px">
        ⚕️ MediVoice Africa — Powered by Gemma 4 E4B | This is triage guidance only, not a diagnosis.
      </div>
    </div>
    """

    # Generate TTS audio
    try:
        audio = speak(response_text[:500], lang_code)  # limit TTS length
    except Exception:
        audio = None

    return html_card, audio


# Build Gradio interface
with gr.Blocks(
    title="🌍 MediVoice Africa",
    theme=gr.themes.Soft(primary_hue="green"),
    css="""
    .gradio-container { max-width: 800px !important; margin: 0 auto; }
    #header { text-align: center; padding: 20px 0 10px; }
    #header h1 { font-size: 2.2em; margin: 0; }
    #header p  { color: #666; margin: 6px 0 0; }
    """
) as demo:

    gr.HTML("""
    <div id="header">
      <h1>🌍 MediVoice Africa</h1>
      <p>Multilingual Medical Triage Assistant · Powered by Gemma 4 E4B · Works Offline</p>
      <p style="font-size:13px;color:#999">
        10 languages · Audio response · No internet needed on device · Built for underserved communities
      </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            lang_dd = gr.Dropdown(
                choices     = LANG_OPTIONS,
                value       = "English",
                label       = "🌐 Your Language",
                interactive = True,
            )
            symptom_box = gr.Textbox(
                label       = "💬 Describe your symptoms",
                placeholder = "Example: I am 45 years old. I have chest pain, sweating, and shortness of breath.",
                lines       = 5,
                max_lines   = 10,
            )

            with gr.Row():
                clear_btn  = gr.Button("🗑️ Clear",  variant="secondary", size="sm")
                submit_btn = gr.Button("🔍 Get Triage", variant="primary",   size="lg")

            gr.Examples(
                examples=[
                    ["English",            "I am 55. I have chest pain spreading to my left arm and I am sweating a lot."],
                    ["Français (French)",  "J'ai 28 ans et j'ai une forte fièvre depuis 5 jours avec de la toux verte."],
                    ["Kiswahili",          "Mtoto wangu ana homa ya juu sana kwa siku 3."],
                    ["हिन्दी (Hindi)",           "मेरी उम्र 42 साल है। पेट के दाहिनी तरफ तेज दर्द है।"],
                    ["العربية (Arabic)",   "عمري 30 سنة. لدي صداع خفيف منذ يوم واحد بعد نوم سيء."],
                    ["Hausa",              "Ina shekara 50. Ina da zazzabi, sanyi, ciwon jiki, da zufa."],
                ],
                inputs=[lang_dd, symptom_box],
                label="📋 Example Queries",
            )

        with gr.Column(scale=1):
            output_html  = gr.HTML(label="Triage Result")
            output_audio = gr.Audio(label="🔊 Listen to Response", autoplay=False)

    submit_btn.click(
        fn      = run_triage,
        inputs  = [lang_dd, symptom_box],
        outputs = [output_html, output_audio],
    )
    clear_btn.click(
        fn=lambda: ("", None),
        outputs=[symptom_box, output_html]
    )

# Launch — share=True gives a public URL for the live demo requirement
# Removed hardcoded ports to prevent "Address already in use" errors on Kaggle
demo.launch(share=True)

# %% [markdown]
# ## Step 7 — Export Results for Writeup

# %%
import datetime

summary = {
    "project": "MediVoice Africa",
    "model": model_path,
    "languages_supported": list(LANG_CODES.values()),
    "demo_results": [
        {
            "lang": r["lang"],
            "triage_level": r["result"].get("triage_level"),
            "confidence": r["result"].get("confidence"),
        }
        for r in results
    ],
    "timestamp": datetime.datetime.utcnow().isoformat(),
}

out_path = "/kaggle/working/medivoice_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"✅ Results saved to {out_path}")
print(json.dumps(summary, indent=2))


In [8]:
# ── Create submission file (required by Kaggle competition) ──────────────────
import csv, os

submission_path = "/kaggle/working/submission.csv"
with open(submission_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["project", "model", "demo_url", "hf_model", "languages"])
    writer.writerow([
        "MediVoice Africa",
        "google/gemma-4-e2b-it fine-tuned with Unsloth QLoRA",
        "https://17727a9e167104dd14.gradio.live",
        "https://huggingface.co/abdokamal/medivoice-africa-gemma4-e2b",
        "en,fr,sw,hi,ar,ha,pt,es,bn,am"
    ])
print(f"✅ submission.csv created ({os.path.getsize(submission_path)} bytes)")


✅ submission.csv created (245 bytes)
